# COGS 138 — Neural Data Science
## Group 7 Final Project
### Are electrophysiological features from the Allen Cell Types Database reflected in resting-state EEG biomarkers for MDD?

**Group Members:** Jessica Truong, Siddhant Gulati, Tony Zheng  
**Dataset 1:** Allen Cell Types Database (mouse cortical neurons)  
**Dataset 2:** OpenNeuro ds003478 — EEG Depression Rest (122 participants)  

---
### Project Overview
This notebook investigates whether micro-scale electrophysiological features (firing rate, action potential amplitude, inter-spike interval) from mouse cortical neurons in the Allen Cell Types Database correlate with macro-scale EEG biomarkers (alpha power, frontal asymmetry) used to decode Major Depressive Disorder (MDD) in humans.

**Dataset Citation:** James F Cavanagh (2021). EEG: Depression rest. OpenNeuro. doi: 10.18112/openneuro.ds003478.v1.1.0


---
## 0. Install & Import Libraries

In [ ]:
# ============================================================
# SETUP: Mount Google Drive + Install Packages
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

%pip install allensdk --quiet
%pip install mne --quiet
%pip install scipy scikit-learn seaborn statsmodels --quiet

import os

# ============================================================
# SET YOUR DATA PATH
# Update this to match where you downloaded the dataset
# ============================================================
DATA_PATH = '/content/drive/MyDrive/Cogs138/ds003478-download/'

# Download dataset only if not already downloaded
if not os.path.exists(DATA_PATH):
    print('Downloading OpenNeuro dataset... (~10-15 mins)...')
    os.makedirs(DATA_PATH, exist_ok=True)
    !aws s3 sync --no-sign-request s3://openneuro.org/ds003478 {DATA_PATH}
    print('Download complete!')
else:
    files = os.listdir(DATA_PATH)
    print(f'Dataset already downloaded! Found {len(files)} items at {DATA_PATH}')


In [ ]:
# Run this cell first to install required packages


# Then update your data path to:
DATA_PATH = '/content/drive/MyDrive/Cogs138/ds003478-download/'
%pip install allensdk 
%pip install mne 
%pip install scipy scikit-learn seaborn 

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# Allen SDK
from allensdk.core.cell_types_cache import CellTypesCache
from allensdk.ephys.ephys_extractor import EphysSweepFeatureExtractor

# EEG processing
import mne

# ML
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

# Settings
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('All libraries loaded successfully!')

---
## PART 1: Allen Cell Types Database
### 1.1 Load Cell Data via SDK

In [ ]:
# Initialize the cache — data will download to 'cell_types/' folder
ctc = CellTypesCache(manifest_file='cell_types/manifest.json')

# Get all cells with human-readable metadata
cells = ctc.get_cells()
cells_df = pd.DataFrame(cells)

print(f'Total cells in database: {len(cells_df)}')
print(f'\nColumns available:')
print(cells_df.columns.tolist())

In [ ]:
# Preview the dataframe
cells_df.head()

### 1.2 Extract Electrophysiological Features

In [ ]:
# Get electrophysiology features for all cells
ephys_features = ctc.get_ephys_features()
ephys_df = pd.DataFrame(ephys_features)

print(f'Ephys feature records: {len(ephys_df)}')
print(f'\nKey features available:')

# Features of interest for our project
features_of_interest = [
    'specimen_id',
    'avg_firing_rate',           # Average firing rate (Hz)
    'f_i_curve_slope',           # Firing rate vs current slope
    'adaptation',                # Spike frequency adaptation
    'mean_isi',                  # Mean inter-spike interval
    'cv_isi',                    # Coefficient of variation of ISI
    'peak_v_long_square',        # Action potential peak voltage (amplitude)
    'trough_v_long_square',      # AP trough voltage
    'fast_trough_v_long_square', # Fast trough
    'upstroke_downstroke_ratio_long_square',  # AP shape
    'input_resistance',          # Input resistance (MOhm)
    'tau',                       # Membrane time constant
    'threshold_v_long_square',   # Spike threshold
]

# Keep only features that exist in the dataframe
available_features = [f for f in features_of_interest if f in ephys_df.columns]
ephys_df = ephys_df[available_features]

print(ephys_df.columns.tolist())
ephys_df.head()

### 1.3 Filter for Cortical Cell Types (Relevant to Depression)

In [ ]:
# Merge cell metadata with ephys features
merged_df = cells_df.merge(ephys_df, left_on='id', right_on='specimen_id', how='inner')

print(f'Merged records: {len(merged_df)}')

# Filter for mouse cortical neurons (most relevant to EEG comparison)
# Focus on visual cortex and prefrontal-like regions
cortical_df = merged_df[merged_df['species'].str.contains('Mus', na=False)].copy()

print(f'Mouse cortical neurons: {len(cortical_df)}')

# Check cell types available
if 'line_name' in cortical_df.columns:
    print(f'\nUnique cell lines: {cortical_df["line_name"].nunique()}')

### 1.4 Exploratory Analysis — Allen Cell Types

In [ ]:
# Distribution of key electrophysiological features
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Allen Cell Types: Distribution of Key Electrophysiological Features', fontsize=14)

plot_features = [
    ('avg_firing_rate', 'Average Firing Rate (Hz)'),
    ('mean_isi', 'Mean ISI (ms)'),
    ('peak_v_long_square', 'AP Peak Voltage (mV)'),
    ('input_resistance', 'Input Resistance (MΩ)'),
    ('tau', 'Membrane Time Constant (ms)'),
    ('adaptation', 'Spike Frequency Adaptation'),
]

for ax, (feat, label) in zip(axes.flatten(), plot_features):
    if feat in cortical_df.columns:
        data = cortical_df[feat].dropna()
        ax.hist(data, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
        ax.set_xlabel(label, fontsize=10)
        ax.set_ylabel('Count')
        ax.set_title(f'{label}\n(n={len(data)})')
    else:
        ax.set_visible(False)

plt.tight_layout()
plt.savefig('allen_features_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap of Allen features
feature_cols = [f for f, _ in plot_features if f in cortical_df.columns]
corr_matrix = cortical_df[feature_cols].corr(method='spearman')

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Spearman Correlation — Allen Cell Electrophysiological Features', fontsize=12)
plt.tight_layout()
plt.savefig('allen_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## PART 2: MODMA EEG Dataset
### 2.1 Load EEG Data

> **Note:** Download the MODMA dataset from http://modma.lzu.edu.cn/data/index/  
> Place the EEG files in a folder called `modma_data/` in the same directory as this notebook.  
> If using the OpenNeuro alternative: https://openneuro.org/datasets/ds003478

In [ ]:
import os

# --------------------------------------------------------
# SET YOUR DATA PATH HERE
# --------------------------------------------------------
DATA_PATH = '/content/drive/MyDrive/ds003478-download/'
  # <-- change this to your actual path

# Check if data exists
if os.path.exists(DATA_PATH):
    files = os.listdir(DATA_PATH)
    print(f'Found {len(files)} files in {DATA_PATH}')
    print(files[:10])
else:
    print('⚠️  Data folder not found!')
    print('Please download MODMA dataset and update DATA_PATH above.')
    print('Download: http://modma.lzu.edu.cn/data/index/')

In [ ]:
def load_eeg_file(filepath):
    """
    Load a single EEG file.
    OpenNeuro ds003478 uses EEGLAB .set format.
    """
    ext = os.path.splitext(filepath)[1].lower()
    if ext == '.set':
        raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
    elif ext == '.edf':
        raw = mne.io.read_raw_edf(filepath, preload=True, verbose=False)
    elif ext == '.bdf':
        raw = mne.io.read_raw_bdf(filepath, preload=True, verbose=False)
    else:
        raise ValueError(f'Unsupported file format: {ext}')
    return raw


def load_participants_labels(data_path):
    """
    Load MDD vs healthy labels from BIDS participants.tsv
    This file is in the root of the OpenNeuro dataset.
    """
    tsv_path = os.path.join(data_path, 'participants.tsv')
    if os.path.exists(tsv_path):
        participants = pd.read_csv(tsv_path, sep='\t')
        print('participants.tsv loaded!')
        print(participants.head(10))
        print(f'Columns: {participants.columns.tolist()}')
        print(f'Total subjects: {len(participants)}')
        return participants
    else:
        print(f'Warning: participants.tsv not found at {tsv_path}')
        print('Check your DATA_PATH is correct!')
        return None


def get_eeg_files(data_path):
    """
    Walk BIDS directory and collect all EEG files.
    OpenNeuro structure: sub-XXX/eeg/sub-XXX_task-Rest_run-01_eeg.set
    """
    eeg_files = []
    for root, dirs, files in os.walk(data_path):
        for f in files:
            if f.endswith('_eeg.set') or f.endswith('_eeg.edf'):
                eeg_files.append(os.path.join(root, f))
    print(f'Found {len(eeg_files)} EEG files')
    return sorted(eeg_files)


# Load labels and file list
participants_df = load_participants_labels(DATA_PATH)
eeg_files_list  = get_eeg_files(DATA_PATH)


### 2.2 Preprocessing Pipeline

In [ ]:
def preprocess_eeg(raw, l_freq=1.0, h_freq=40.0, notch_freq=60.0):
    """
    Standard EEG preprocessing pipeline:
    1. Bandpass filter (1-40 Hz)
    2. Notch filter (60 Hz US powerline)
    3. Re-reference to average
    
    Parameters
    ----------
    raw : mne.io.Raw
    l_freq : float, lower bandpass frequency
    h_freq : float, upper bandpass frequency  
    notch_freq : float, powerline noise frequency
    
    Returns
    -------
    raw_filtered : preprocessed MNE Raw object
    """
    raw_filtered = raw.copy()
    
    # Bandpass filter
    raw_filtered.filter(l_freq=l_freq, h_freq=h_freq, verbose=False)
    
    # Notch filter for powerline noise
    raw_filtered.notch_filter(freqs=notch_freq, verbose=False)
    
    # Re-reference to average
    raw_filtered.set_eeg_reference('average', projection=False, verbose=False)
    
    return raw_filtered

print('Preprocessing function defined.')

### 2.3 Extract EEG Features: Alpha Power & Frontal Asymmetry

In [ ]:
def compute_band_power(raw, band=(8, 12), frontal_channels=None):
    """
    Compute average power in a frequency band using Welch's method.
    
    Parameters
    ----------
    raw : mne.io.Raw
    band : tuple, (low_freq, high_freq) in Hz
    frontal_channels : list of channel names (e.g. ['F3', 'F4', 'Fz'])
    
    Returns
    -------
    band_powers : dict of {channel: power}
    """
    from mne.time_frequency import psd_array_welch
    
    sfreq = raw.info['sfreq']
    
    if frontal_channels:
        raw_pick = raw.copy().pick_channels(frontal_channels, ordered=True)
    else:
        raw_pick = raw.copy()
    
    data = raw_pick.get_data()
    
    psds, freqs = psd_array_welch(
        data,
        sfreq=sfreq,
        fmin=band[0],
        fmax=band[1],
        n_fft=int(sfreq * 2),  # 2-second windows
        verbose=False
    )
    
    # Average power across frequency band
    band_power = psds.mean(axis=1)  # shape: (n_channels,)
    
    ch_names = raw_pick.ch_names
    return dict(zip(ch_names, band_power))


def compute_frontal_alpha_asymmetry(raw, left_ch='F3', right_ch='F4'):
    """
    Compute Frontal Alpha Asymmetry (FAA).
    FAA = ln(right_alpha) - ln(left_alpha)
    Positive FAA = relative left withdrawal (associated with depression)
    
    Parameters
    ----------
    raw : mne.io.Raw
    left_ch : str, left frontal channel name
    right_ch : str, right frontal channel name
    
    Returns
    -------
    faa : float
    """
    try:
        powers = compute_band_power(raw, band=(8, 12),
                                     frontal_channels=[left_ch, right_ch])
        left_alpha = powers.get(left_ch, np.nan)
        right_alpha = powers.get(right_ch, np.nan)
        
        faa = np.log(right_alpha) - np.log(left_alpha)
        return faa
    except Exception as e:
        print(f'FAA computation error: {e}')
        return np.nan


def extract_all_eeg_features(raw):
    """
    Extract all EEG features for one subject.
    
    Returns
    -------
    features : dict
    """
    features = {}
    
    # Alpha band power (8-12 Hz) — averaged across all channels
    alpha_powers = compute_band_power(raw, band=(8, 12))
    features['mean_alpha_power'] = np.mean(list(alpha_powers.values()))
    
    # Theta band power (4-8 Hz)
    theta_powers = compute_band_power(raw, band=(4, 8))
    features['mean_theta_power'] = np.mean(list(theta_powers.values()))
    
    # Beta band power (12-30 Hz)
    beta_powers = compute_band_power(raw, band=(12, 30))
    features['mean_beta_power'] = np.mean(list(beta_powers.values()))
    
    # Frontal alpha asymmetry
    features['frontal_alpha_asymmetry'] = compute_frontal_alpha_asymmetry(raw)
    
    # Theta/alpha ratio (elevated in depression)
    features['theta_alpha_ratio'] = features['mean_theta_power'] / (features['mean_alpha_power'] + 1e-10)
    
    return features

print('EEG feature extraction functions defined.')

In [ ]:
def process_all_subjects(data_path, label_file=None):
    """
    Process all EEG files in a directory and extract features.
    
    Parameters
    ----------
    data_path : str, path to EEG files
    label_file : str (optional), path to CSV with subject labels
                 Expected columns: ['subject_id', 'label'] where label: 1=MDD, 0=healthy
    
    Returns
    -------
    features_df : DataFrame with EEG features per subject
    """
    results = []
    eeg_extensions = ['.edf', '.bdf', '.set']
    
    if not os.path.exists(data_path):
        print(f'Data path {data_path} not found. Using synthetic data for demonstration.')
        return None
    
    files = [f for f in os.listdir(data_path) 
             if any(f.endswith(ext) for ext in eeg_extensions)]
    
    print(f'Processing {len(files)} EEG files...')
    
    for i, fname in enumerate(files):
        try:
            fpath = os.path.join(data_path, fname)
            raw = load_eeg_file(fpath)
            raw = preprocess_eeg(raw)
            feats = extract_all_eeg_features(raw)
            feats['subject_id'] = fname.replace('.edf', '').replace('.bdf', '').replace('.set', '')
            results.append(feats)
            
            if (i + 1) % 10 == 0:
                print(f'Processed {i + 1}/{len(files)} files...')
                
        except Exception as e:
            print(f'Error processing {fname}: {e}')
            continue
    
    features_df = pd.DataFrame(results)
    
    # Merge with labels if provided
    if label_file and os.path.exists(label_file):
        labels_df = pd.read_csv(label_file)
        features_df = features_df.merge(labels_df, on='subject_id', how='left')
    
    return features_df


# Run processing (comment out if data not yet downloaded)
# eeg_features_df = process_all_subjects(DATA_PATH, label_file='modma_data/labels.csv')

print('Processing function defined. Uncomment last line when data is downloaded.')

### 2.4 Synthetic Data Demo
Run this section to test the pipeline while waiting for the MODMA dataset.

In [ ]:
np.random.seed(42)
n_subjects = 106  # 53 MDD + 53 healthy (MODMA size)
labels = np.array([1]*53 + [0]*53)  # 1=MDD, 0=healthy

# Simulate EEG features with known group differences
# MDD: lower alpha, higher theta, more asymmetry (based on literature)
eeg_features_df = pd.DataFrame({
    'subject_id': [f'sub_{i:03d}' for i in range(n_subjects)],
    'label': labels,
    'mean_alpha_power': np.concatenate([
        np.random.normal(0.8, 0.2, 53),   # MDD: lower alpha
        np.random.normal(1.2, 0.2, 53)    # Healthy: higher alpha
    ]),
    'mean_theta_power': np.concatenate([
        np.random.normal(1.3, 0.3, 53),   # MDD: higher theta
        np.random.normal(0.9, 0.2, 53)    # Healthy: lower theta
    ]),
    'mean_beta_power': np.concatenate([
        np.random.normal(0.6, 0.15, 53),
        np.random.normal(0.7, 0.15, 53)
    ]),
    'frontal_alpha_asymmetry': np.concatenate([
        np.random.normal(0.3, 0.15, 53),  # MDD: positive asymmetry
        np.random.normal(0.0, 0.15, 53)   # Healthy: near zero
    ]),
    'theta_alpha_ratio': np.concatenate([
        np.random.normal(1.7, 0.4, 53),   # MDD: elevated ratio
        np.random.normal(0.8, 0.2, 53)    # Healthy
    ]),
})

print('Synthetic EEG dataset created (replace with real MODMA data):')
print(eeg_features_df.groupby('label').describe().round(3))

### 2.5 Visualize EEG Features by Group

In [ ]:
eeg_feature_cols = ['mean_alpha_power', 'mean_theta_power', 
                     'mean_beta_power', 'frontal_alpha_asymmetry', 'theta_alpha_ratio']

fig, axes = plt.subplots(1, 5, figsize=(18, 5))
fig.suptitle('EEG Features: MDD vs Healthy Controls', fontsize=14, fontweight='bold')

colors = {1: '#E74C3C', 0: '#2ECC71'}  # Red=MDD, Green=Healthy
labels_text = {1: 'MDD', 0: 'Healthy'}

for ax, feat in zip(axes, eeg_feature_cols):
    for label_val, color in colors.items():
        data = eeg_features_df[eeg_features_df['label'] == label_val][feat]
        ax.hist(data, bins=20, alpha=0.6, color=color,
                label=labels_text[label_val], edgecolor='white')
    
    # Statistical test
    mdd_data = eeg_features_df[eeg_features_df['label'] == 1][feat]
    hc_data = eeg_features_df[eeg_features_df['label'] == 0][feat]
    t_stat, p_val = stats.ttest_ind(mdd_data, hc_data)
    
    ax.set_title(f'{feat.replace("_", " ").title()}\np={p_val:.3f}', fontsize=9)
    ax.legend(fontsize=8)
    ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig('eeg_features_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## PART 3: Decoding — MDD Classification from EEG
### 3.1 Build and Evaluate Classifier

In [ ]:
# Prepare features and labels
X = eeg_features_df[eeg_feature_cols].values
y = eeg_features_df['label'].values

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Cross-validated logistic regression
clf = LogisticRegression(random_state=42, max_iter=1000)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(clf, X_scaled, y, cv=cv, scoring='accuracy')

print('=== MDD Decoding Results ===')
print(f'Cross-validated accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
print(f'Individual fold scores: {cv_scores.round(3)}')
print(f'\nChance level: 0.500')
print(f'Above chance: {cv_scores.mean() > 0.5}')

In [ ]:
# Fit final model and inspect feature importance
clf.fit(X_scaled, y)

coef_df = pd.DataFrame({
    'Feature': eeg_feature_cols,
    'Coefficient': clf.coef_[0],
    'Abs_Coefficient': np.abs(clf.coef_[0])
}).sort_values('Abs_Coefficient', ascending=True)

plt.figure(figsize=(8, 4))
colors_bar = ['#E74C3C' if c > 0 else '#3498DB' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors_bar, edgecolor='white')
plt.axvline(x=0, color='black', linewidth=0.8)
plt.xlabel('Logistic Regression Coefficient')
plt.title('EEG Feature Importance for MDD Decoding\n(Red = predicts MDD, Blue = predicts Healthy)')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## PART 4: Cross-Scale Correlation
### 4.1 Compute Summary Statistics from Allen Data

In [ ]:
# Compute population-level summary statistics from Allen Cell Types
# These represent the 'micro-scale' reference features

allen_numeric_cols = [f for f, _ in plot_features if f in cortical_df.columns]

allen_summary = cortical_df[allen_numeric_cols].describe().T[['mean', 'std', '50%']]
allen_summary.columns = ['allen_mean', 'allen_std', 'allen_median']

print('Allen Cell Types — Population Summary Statistics:')
print(allen_summary.round(4))

### 4.2 Feature Mapping: Micro → Macro

In [ ]:
# Conceptual mapping between Allen features and EEG features
# Based on literature linking single-neuron properties to population oscillations

feature_mapping = {
    'avg_firing_rate':     'mean_alpha_power',         # Higher firing → higher alpha
    'mean_isi':            'mean_theta_power',          # Longer ISI → more theta
    'peak_v_long_square':  'mean_beta_power',           # AP amplitude → beta power
    'adaptation':          'frontal_alpha_asymmetry',   # Adaptation → asymmetry
    'input_resistance':    'theta_alpha_ratio',         # Resistance → theta/alpha
}

print('Proposed Feature Mapping (Micro → Macro):')
print('=' * 60)
for allen_feat, eeg_feat in feature_mapping.items():
    print(f'  Allen: {allen_feat:<35} → EEG: {eeg_feat}')

In [ ]:
# Simulate subject-level Allen-like features for correlation analysis
# In real analysis: use Allen SDK to get per-cell-type distributions
# and map to subjects via literature-based priors

np.random.seed(42)

# Simulate micro-scale features per subject (proxy from Allen distributions)
allen_proxy = pd.DataFrame({
    'subject_id': eeg_features_df['subject_id'],
    'label': eeg_features_df['label'],
    # MDD associated with lower firing rates, longer ISI (literature-based)
    'avg_firing_rate': np.concatenate([
        np.random.normal(4.0, 1.5, 53),
        np.random.normal(6.5, 1.5, 53)
    ]),
    'mean_isi': np.concatenate([
        np.random.normal(280, 60, 53),
        np.random.normal(200, 50, 53)
    ]),
    'peak_v_long_square': np.concatenate([
        np.random.normal(38, 5, 53),
        np.random.normal(42, 4, 53)
    ]),
})

# Merge with EEG features
combined_df = eeg_features_df.merge(allen_proxy, on=['subject_id', 'label'])
print(f'Combined dataset shape: {combined_df.shape}')
combined_df.head()

### 4.3 Spearman Correlation Analysis

In [ ]:
# Compute Spearman correlations between Allen features and EEG features
allen_feat_cols = ['avg_firing_rate', 'mean_isi', 'peak_v_long_square']
eeg_feat_cols_sub = ['mean_alpha_power', 'mean_theta_power', 
                      'frontal_alpha_asymmetry', 'theta_alpha_ratio']

corr_results = []

for af in allen_feat_cols:
    for ef in eeg_feat_cols_sub:
        rho, p = spearmanr(combined_df[af], combined_df[ef])
        corr_results.append({
            'Allen Feature': af,
            'EEG Feature': ef,
            'Spearman_rho': rho,
            'p_value': p
        })

corr_df = pd.DataFrame(corr_results)

# FDR correction
_, p_corrected, _, _ = multipletests(corr_df['p_value'], method='fdr_bh')
corr_df['p_corrected'] = p_corrected
corr_df['significant'] = corr_df['p_corrected'] < 0.05

print('Spearman Correlation Results (with FDR correction):')
print(corr_df.round(4).to_string())

In [ ]:
# Visualize as a heatmap
corr_pivot = corr_df.pivot(index='Allen Feature', 
                            columns='EEG Feature', 
                            values='Spearman_rho')

sig_pivot = corr_df.pivot(index='Allen Feature',
                           columns='EEG Feature',
                           values='significant')

plt.figure(figsize=(9, 4))
ax = sns.heatmap(corr_pivot, annot=True, fmt='.3f', cmap='coolwarm',
                  center=0, vmin=-1, vmax=1, linewidths=0.5,
                  cbar_kws={'label': 'Spearman ρ'})

# Mark significant correlations with asterisk
for i in range(sig_pivot.shape[0]):
    for j in range(sig_pivot.shape[1]):
        if sig_pivot.iloc[i, j]:
            ax.text(j + 0.8, i + 0.2, '*', fontsize=14, color='black', fontweight='bold')

plt.title('Cross-Scale Correlation: Allen Cell Features vs EEG Biomarkers\n(* = significant after FDR correction)', 
          fontsize=11)
plt.tight_layout()
plt.savefig('cross_scale_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## PART 5: Summary & Conclusions

In [ ]:
print('=' * 60)
print('COGS 138 GROUP 7 — PROJECT SUMMARY')
print('=' * 60)

print(f'''
Research Question:
Are electrophysiological features from the Allen Cell Types Database 
reflected in resting-state EEG biomarkers for MDD?

Datasets:
  - Allen Cell Types Database (n = {len(cortical_df) if len(cortical_df) > 0 else 'TBD'} mouse cortical neurons)
  - MODMA EEG Dataset (n = {len(eeg_features_df)} subjects: 53 MDD, 53 healthy)

Key Findings:
  1. EEG Decoding Accuracy:   {cv_scores.mean():.1%} ± {cv_scores.std():.1%}
  2. Most predictive EEG feature: {coef_df.iloc[-1]["Feature"]}
  3. Significant cross-scale correlations: {corr_df["significant"].sum()} / {len(corr_df)}

Limitations:
  - Cross-species comparison (mouse → human)
  - Different measurement scales (single neuron → scalp EEG)
  - Synthetic data used for demonstration (replace with real MODMA)

Next Steps:
  - Download and process real MODMA EEG data
  - Refine Allen cell type filtering (focus on PFC-relevant cells)
  - Add ICA artifact removal to EEG preprocessing
  - Try non-linear classifiers (SVM, Random Forest)
''')

print('=' * 60)

---
## References

1. Cavanagh, J. F. (2021). EEG: Depression rest. *OpenNeuro.* doi: 10.18112/openneuro.ds003478.v1.1.0
2. Cavanagh, J. F., et al. (2019). Anger releases behavioral inhibition to cascade across response boundaries. *Psychophysiology.* PMID: 31149639
3. Quinn, G. M. V. (2024). Resting-state EEG microstate features for MDD classification. *CUNY Academic Works.*
4. Siebenbühner, F., et al. (2026). Brain anatomy and molecular signaling predict neurophysiological dynamics. *bioRxiv.*
5. Gouwens, N. W., et al. (2019). Classification of electrophysiological and morphological neuron types in mouse visual cortex. *Nature Neuroscience.*
6. Wu, X., et al. (2021). Resting-state EEG signal for MDD detection. *Biosensors, MDPI.*
7. Yang, et al. (2023). Depression detection based on EEG signals in multi brain regions. *Journal of Integrative Neuroscience.*
